# Testing con pytest

Escribir código que *parece* funcionar es fácil. Saber que **sigue** funcionando
después de cada cambio es otra cosa. Para eso existen los **tests**: código que
comprueba automáticamente que nuestro programa hace lo que debe.

> 🎯 **Un test es una red de seguridad.** Te permite cambiar y mejorar el código sin
> miedo, porque si rompes algo, los tests te avisan al instante.

Este vídeo es un poco distinto a los demás: en vez de ir ejecutando celdas, vamos a
trabajar con un **mini-proyecto real**, con sus archivos `.py`, y ejecutaremos los
tests **desde la terminal**, que es como se hace en el día a día. Este notebook es la
**guía**: explica qué hay en cada archivo y cómo usarlo.

Usaremos `pytest`, el framework de testing más popular de Python por su sencillez.

> 📚 Documentación de referencia: **[pytest](https://docs.pytest.org/)**.

---

## 1. ¿Por qué testear?

Cuando escribimos una función, solemos probarla "a mano": la ejecutamos con un par de
valores, miramos si el resultado tiene buena pinta y seguimos. El problema es que esa
comprobación **no queda guardada**, **no se repite sola** y suele ser **incompleta**
(es fácil olvidar los casos raros: el cero, el negativo, la lista vacía...).

Los tests resuelven esto: son comprobaciones **escritas una vez** que se ejecutan
**automáticamente** tantas veces como quieras. Sus ventajas:

- **Confianza para cambiar código** (*refactorizar*) sin romper nada sin darte cuenta.
- **Detección temprana** de errores: antes de que lleguen al usuario.
- **Documentación viva**: un test muestra *cómo se usa* y *qué se espera* del código.

La forma más básica de comprobar algo es la sentencia `assert`: no hace nada si la
condición es verdadera, pero lanza un `AssertionError` si es falsa.

```python
cuenta = Cuenta("Ana", 100)
cuenta.depositar(50)
assert cuenta.saldo == 150   # si fuera falso, saltaría AssertionError
```

Esto está bien para una comprobación suelta, pero no escala: si un `assert` falla, se
detiene todo; no hay informe de "cuántos pasaron y cuántos fallaron"; y organizar
muchos a mano se vuelve un lío. Para eso usamos un **framework**: `pytest`.

---

## 2. La estructura del proyecto

Abre la carpeta `16_testing_pytest/` en tu editor. Contiene:

```
16_testing_pytest/
├── 16_testing_pytest.ipynb   ← este notebook (la guía)
├── cuenta.py                 ← el código que queremos probar
├── conftest.py               ← fixtures compartidas (datos de prueba)
└── test_cuenta.py            ← los tests de la cuenta
```

Esta separación es la convención habitual: el **código de producción** por un lado
(`cuenta.py`) y los **tests** por otro (`test_cuenta.py`). pytest reconoce como tests
los archivos que empiezan por `test_` y, dentro, las funciones que empiezan por
`test_`.

> 💡 Te recomiendo tener el editor y una terminal abiertos a la vez: vamos a editar
> los `.py` y a lanzar `pytest` en la terminal para ver el resultado.

---

## 3. El código que vamos a probar: `cuenta.py`

Nuestro "código de producción" es una clase `Cuenta` (una cuenta bancaria) con sus
**reglas de negocio**: no se puede retirar más de lo que hay, las cantidades deben ser
positivas y el saldo inicial no puede ser negativo.

```python
# cuenta.py
class SaldoInsuficiente(Exception):
    """Se lanza al intentar retirar más dinero del disponible."""


class Cuenta:
    def __init__(self, titular, saldo=0):
        if saldo < 0:
            raise ValueError("El saldo inicial no puede ser negativo")
        self.titular = titular
        self.saldo = saldo

    def depositar(self, cantidad):
        if cantidad <= 0:
            raise ValueError("La cantidad debe ser positiva")
        self.saldo += cantidad

    def retirar(self, cantidad):
        if cantidad <= 0:
            raise ValueError("La cantidad debe ser positiva")
        if cantidad > self.saldo:
            raise SaldoInsuficiente("Fondos insuficientes")
        self.saldo -= cantidad
```

Cada una de esas reglas es algo que podemos (y debemos) **comprobar con un test**.

---

## 4. Instalar pytest

`pytest` no viene incluido en Python; se instala como **dependencia de desarrollo**
(solo lo necesitamos mientras desarrollamos, no en producción), como vimos con otras
herramientas del curso:

```bash
uv add --dev pytest
```

Para comprobar que está disponible:

```bash
pytest --version
```

---

## 5. El primer test: `assert` de toda la vida

La gran ventaja de pytest es que los tests son **funciones normales** que usan el
`assert` de siempre. No hacen falta clases ni métodos especiales. Mira el principio de
`test_cuenta.py`:

```python
# test_cuenta.py
from cuenta import Cuenta, SaldoInsuficiente


def test_depositar_aumenta_saldo(cuenta):
    cuenta.depositar(50)
    assert cuenta.saldo == 150
```

Dos cosas a destacar:

- El cuerpo del test es Python normal y un `assert`. Cuando el `assert` falla, pytest
  es muy listo y te muestra **los valores concretos** que no cuadraban (no solo "falló").
- El parámetro `cuenta` no sale de la nada: es una **fixture** (lo vemos en la sección
  8). pytest se encarga de crear una cuenta y pasársela al test.

---

## 6. Ejecutar los tests desde la terminal

Esta es la parte importante. Abre una terminal, **sitúate dentro de la carpeta**
`16_testing_pytest/` y ejecuta:

```bash
pytest
```

pytest **descubre solo** los tests y los ejecuta. Cada punto `.` es un test que pasa:

```
============================= test session starts =============================
collected 9 items

test_cuenta.py .........                                                 [100%]

============================== 9 passed in 0.06s ==============================
```

Para ver **un renglón por test** (más útil para entender qué se está probando), usa el
modo *verbose* con `-v`:

```bash
pytest -v
```

```
test_cuenta.py::test_saldo_inicial PASSED                                [ 11%]
test_cuenta.py::test_depositar_aumenta_saldo PASSED                      [ 22%]
test_cuenta.py::test_retirar_disminuye_saldo PASSED                      [ 33%]
test_cuenta.py::test_retirar_de_mas_lanza_error PASSED                   [ 44%]
test_cuenta.py::test_depositar_cantidad_negativa_lanza_error PASSED      [ 55%]
test_cuenta.py::test_saldo_inicial_negativo_lanza_error PASSED           [ 66%]
test_cuenta.py::test_depositar_parametrizado[100-50-150] PASSED          [ 77%]
test_cuenta.py::test_depositar_parametrizado[0-10-10] PASSED             [ 88%]
test_cuenta.py::test_depositar_parametrizado[200-1-201] PASSED           [100%]
```

> 💡 Prueba a romper algo a propósito (por ejemplo, cambia en `cuenta.py` el
> `self.saldo += cantidad` por `-=`) y vuelve a ejecutar `pytest`. Verás cómo el test
> correspondiente **falla** y pytest te enseña el valor esperado y el obtenido. Esa es
> exactamente la red de seguridad de la que hablábamos.

---

## 7. Probar que algo falla: `pytest.raises`

Comprobar que el código **falla cuando debe** es tan importante como comprobar que
funciona. Las reglas de negocio ("no se puede retirar de más", "la cantidad debe ser
positiva") también se testean. Para eso usamos `pytest.raises`, que comprueba que un
bloque lanza la excepción esperada:

```python
import pytest
from cuenta import Cuenta, SaldoInsuficiente


def test_retirar_de_mas_lanza_error(cuenta):
    with pytest.raises(SaldoInsuficiente):
        cuenta.retirar(999)


def test_depositar_cantidad_negativa_lanza_error(cuenta):
    with pytest.raises(ValueError):
        cuenta.depositar(-10)
```

Si el bloque **no** lanza la excepción esperada, el test falla. Es la forma de dejar
claro (y garantizado) que tus validaciones realmente saltan.

---

## 8. Fixtures: preparar el escenario (`conftest.py`)

¿Te has fijado en que muchos tests reciben un parámetro `cuenta` pero nunca creamos esa
cuenta dentro del test? Eso es una **fixture**: una función que prepara algo y se lo
"inyecta" a los tests que lo pidan (poniéndolo como parámetro).

Nuestra fixture está en `conftest.py`, un archivo especial que pytest carga solo y cuyas
fixtures quedan disponibles para todos los tests de la carpeta:

```python
# conftest.py
import pytest
from cuenta import Cuenta


@pytest.fixture
def cuenta():
    return Cuenta("Ana", 100)   # escenario de partida
```

La clave: pytest ejecuta la fixture **antes de cada test**, así que cada test recibe
una cuenta **nueva e independiente**. Aunque un test modifique su cuenta, el siguiente
empieza de cero.

> 💡 Esto garantiza que los tests sean **independientes** entre sí. De hecho, pytest
> puede ejecutarlos en cualquier orden: si están bien aislados, siempre dan el mismo
> resultado.

---

## 9. El mismo test con muchos datos: `parametrize`

A menudo queremos probar la misma lógica con **varios casos**. En lugar de copiar y
pegar el test, lo **parametrizamos**: le damos una lista de entradas y resultados
esperados, y pytest ejecuta cada caso como un test independiente.

```python
import pytest
from cuenta import Cuenta


@pytest.mark.parametrize(
    "saldo_inicial, deposito, esperado",
    [
        (100, 50, 150),
        (0, 10, 10),
        (200, 1, 201),
    ],
)
def test_depositar_parametrizado(saldo_inicial, deposito, esperado):
    cuenta = Cuenta("Ana", saldo_inicial)
    cuenta.depositar(deposito)
    assert cuenta.saldo == esperado
```

Esa única función genera **tres tests**. En la salida de `pytest -v` los verás como
`test_depositar_parametrizado[100-50-150]`, etc. Si uno falla, pytest te dice
exactamente **qué combinación** de datos fue la culpable.

---

## 10. Comandos útiles de pytest

Algunos que usarás a diario, todos desde la terminal en la carpeta del proyecto:

| Comando | Para qué |
|---------|----------|
| `pytest` | Ejecutar todos los tests |
| `pytest -v` | Modo detallado: un renglón por test |
| `pytest -k retirar` | Solo los tests cuyo nombre contiene `retirar` |
| `pytest test_cuenta.py` | Solo los tests de ese archivo |
| `pytest test_cuenta.py::test_saldo_inicial` | Un único test concreto |
| `pytest -x` | Parar en el primer fallo |
| `pytest -q` | Salida más corta (*quiet*) |

---

## 11. Buenas prácticas al escribir tests

Un buen test no es solo "código que comprueba algo". Estas pautas marcan la diferencia:

- **Patrón AAA (*Arrange – Act – Assert*)**: estructura cada test en tres bloques:
  **preparar** los datos, **ejecutar** la acción y **comprobar** el resultado. (En
  nuestros tests, el *Arrange* suele hacerlo la fixture `cuenta`.)
- **Nombres descriptivos**: `test_retirar_de_mas_lanza_error` dice qué comprueba mucho
  mejor que `test_1`. El nombre del test es documentación.
- **Un concepto por test**: cada test debería comprobar **una sola cosa**. Si falla,
  sabes exactamente qué se rompió.
- **Prueba los casos límite (*edge cases*)**: el cero, los negativos, la lista vacía,
  el valor máximo... ahí se esconden la mayoría de los bugs.
- **Tests rápidos, aislados y repetibles**: no deben depender de internet, de la hora
  del sistema ni unos de otros.

> 💡 Una guía muy citada son las propiedades **FIRST**: *Fast* (rápidos), *Independent*
> (independientes), *Repeatable* (repetibles), *Self-validating* (pasan o fallan, sin
> interpretación) y *Timely* (escritos a tiempo, junto al código).

---

## 12. Resumen

En este vídeo hemos montado un mini-proyecto y aprendido a testearlo con `pytest`:

1. **Por qué testear**: los tests son una red de seguridad que da confianza para
   cambiar el código, detecta errores pronto y documenta cómo se usa.
2. **Estructura**: el código de producción (`cuenta.py`) separado de los tests
   (`test_cuenta.py`); pytest descubre solo los `test_*.py` y las funciones `test_*`.
3. **Tests con pytest**: funciones normales con `assert` de toda la vida.
4. **`pytest.raises`**: comprobar que el código falla cuando debe.
5. **Fixtures** (`conftest.py`): preparar un escenario limpio e independiente para cada
   test.
6. **`parametrize`**: ejecutar el mismo test con muchos datos.
7. Se ejecuta **desde la terminal** (`pytest`, `pytest -v`, `pytest -k ...`).
8. **Buenas prácticas**: patrón AAA, nombres descriptivos, casos límite y tests
   rápidos, aislados y repetibles (**FIRST**).

| Archivo / herramienta | Para qué |
|-----------------------|----------|
| `cuenta.py` | El código de producción que queremos comprobar |
| `test_cuenta.py` | Los tests (funciones `test_*` con `assert`) |
| `conftest.py` | Fixtures compartidas (escenario de prueba) |
| `pytest` (terminal) | Descubre y ejecuta los tests |
| `pytest.raises` | Comprobar que algo falla cuando debe |
| `@parametrize` | El mismo test con muchos datos |
| `pytest-cov` | Ver qué código ejecutan los tests |

Empezar a testear cambia la forma de programar: dejas de "cruzar los dedos" al tocar el
código y empiezas a confiar en una red que te avisa al instante si algo se rompe.